<a href="https://colab.research.google.com/github/dilbal/db125msc26project/blob/main/SalasMolina2025_Replication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Salas-Molina et al. (2025): Replication Study

**Question:** How do six distance measures affect HRP return and variance estimation errors across Bull, Bear and Sideways samples?

The notebook uses Yahoo Finance data for historical IBEX candidates and the IBEX index. It is an approximate replication: the available universe, five-fold choice, VI binning and ESR optimizer are implementation choices.

## Notebook map

1. Setup, configuration and data
2. HRP, distance measures and benchmark methods
3. Contiguous-fold evaluation
4. Run the three scenarios
5. Tables and visual comparison
6. Interpretation and methodological notes
7. Historical findings

**Run order:** execute from top to bottom in a fresh runtime. Downloads and optimization can take time.

**Reading the output:** Return and Risk are normalized estimation errors, not portfolio return and volatility. Lower NMSE means a smaller train-versus-test discrepancy relative to the index.


## 1. Setup

Install the download package and import the libraries for data handling, clustering, optimization and plotting. The current code suppresses warnings; this does not establish optimizer convergence.


In [ ]:
# Import the shared libraries for data, portfolio construction and plotting.

!pip install yfinance -q

import numpy as np
import pandas as pd
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import squareform
from scipy.optimize import minimize, differential_evolution
import yfinance as yf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

### 1.1 Configuration

- `K_FOLDS = 5`: number of contiguous held-out blocks.
- `MIN_COVERAGE = 0.95`: approximate minimum price availability within a scenario.
- `SCENARIOS`: the three date ranges used in this replication.
- `IBEX35_TICKERS`: candidate historical symbols, filtered by availability.

The original notes report 25/32/36 available assets versus 38/41/43 in the paper. Treat the downloaded counts as run-specific; the list and filtering do not reconstruct point-in-time index membership.


In [ ]:
# Define scenario ranges, coverage filtering and the candidate symbol list.
# Counts in paper-reference comments are not assertions about the downloaded data.

K_FOLDS = 5        # k for CV (not stated in paper)
MIN_COVERAGE = 0.95

# Table 1 from paper
SCENARIOS = {
    'Bull':     ('2005-01-14', '2007-12-28'),  # 155 weeks, 38 assets in paper
    'Bear':     ('2008-01-06', '2012-08-12'),  # 241 weeks, 41 assets in paper
    'Sideways': ('2014-04-27', '2019-04-07'),  # 259 weeks, 43 assets in paper
}

# Comprehensive historical IBEX 35 constituents (identified from paper Fig 1/2
# + known historical members). Some unavailable on Yahoo Finance.
IBEX35_TICKERS = [
    # Identified from paper Fig 1/2
    'MRL.MC', 'CABK.MC', 'CLNX.MC', 'NTGY.MC', 'MTS.MC',
    'SAB.MC', 'VIS.MC', 'ELE.MC', 'COL.MC', 'ENG.MC',
    'ANA.MC', 'MEL.MC', 'FER.MC', 'MAP.MC', 'ACS.MC',
    'ITX.MC', 'GRF.MC', 'AMS.MC', 'IBE.MC', 'REP.MC',
    'BBVA.MC', 'TEF.MC', 'SAN.MC',
    # Additional historical members
    'ACX.MC', 'BKT.MC', 'FCC.MC', 'OHL.MC', 'POP.MC',
    'BME.MC', 'IDR.MC', 'RED.MC', 'IAG.MC', 'AENA.MC',
    'GAS.MC', 'DIA.MC', 'TRE.MC', 'LOG.MC', 'ROVI.MC',
    'SGRE.MC', 'SLR.MC', 'SOL.MC', 'FDR.MC', 'ABE.MC',
    'NHH.MC', 'EBRO.MC', 'CAF.MC', 'SPS.MC', 'ZOT.MC',
]

### 1.2 Download and align weekly returns

Download adjusted prices, resample to Tuesday-labelled weekly bins, filter sparse stock series and calculate simple returns. The IBEX index is downloaded separately.

**Data handling:** stock prices are forward-filled and then backward-filled. Backward filling uses later prices for leading gaps. Availability filtering also uses the full scenario. These choices are preserved for replication, but the preprocessing is not a strictly past-only trading simulation.


In [ ]:
# Download adjusted prices and form a consistent weekly grid.
# Filter stock coverage before filling missing values and calculating returns.

def download_returns(start, end, tickers, freq='1wk'):
    raw = yf.download(tickers, start=start, end=end, interval=freq,
                      auto_adjust=True, progress=False)
    if raw.empty:
        return pd.DataFrame()
    prices = raw['Close'] if isinstance(raw.columns, pd.MultiIndex) else raw[['Close']]
    if prices.empty:
        return pd.DataFrame()
    # Resample to consistent weekly (some tickers return daily data in batch downloads)
    prices = prices.resample('W-TUE').last()
    prices = prices.loc[start:end]
    # int() rounds the coverage threshold down to a whole observation.
    min_obs = max(1, int(len(prices) * MIN_COVERAGE))
    # Backward filling leading gaps uses later prices.
    prices = prices.dropna(axis=1, thresh=min_obs).ffill().bfill()
    returns = prices.pct_change().dropna().dropna(axis=1)
    return returns

def download_index(start, end, freq='1wk'):
    raw = yf.download('^IBEX', start=start, end=end, interval=freq,
                      auto_adjust=True, progress=False)
    prices = raw['Close'].squeeze() if isinstance(raw.columns, pd.MultiIndex) else raw['Close']
    prices = prices.resample('W-TUE').last().loc[start:end]
    return prices.pct_change().dropna()

print('Downloading IBEX 35 data...')
scenario_returns = {}
scenario_index   = {}
for name, (start, end) in SCENARIOS.items():
    ret = download_returns(start, end, IBEX35_TICKERS)
    idx = download_index(start, end)
    scenario_returns[name] = ret
    scenario_index[name]   = idx
    print(f'  {name}: {ret.shape[1]} assets, {ret.shape[0]} weeks '
          f'(paper: {"38" if name=="Bull" else "41" if name=="Bear" else "43"} assets)')

## 2. Portfolio construction

### 2.1 Shared HRP allocation

* Single linkage creates the hierarchy. `getQuasiDiag` expands it into an asset order; it does not perform clustering itself. `getRecBipart` repeatedly halves that order and allocates more capital to the side with lower inverse-variance portfolio risk.

* All six HRP candidates use the same covariance and allocation machinery; only the distance supplied to clustering changes.


In [ ]:
# Use inverse-variance cluster risk and midpoint bisection for every HRP candidate.

def getIVP(cov, **kargs):
    ivp = 1.0 / np.diag(cov)
    ivp /= ivp.sum()
    return ivp

def getClusterVar(cov, cItems):
    cov_ = cov.loc[cItems, cItems]
    w_ = getIVP(cov_.values).reshape(-1, 1)
    return float(np.dot(np.dot(w_.T, cov_), w_))

def getQuasiDiag(link):
    # Expand merged-cluster nodes into an ordered list of asset positions.
    link = link.astype(int)
    sortIx = pd.Series([link[-1, 0], link[-1, 1]])
    numItems = link[-1, 3]
    while sortIx.max() >= numItems:
        sortIx.index = range(0, sortIx.shape[0] * 2, 2)
        df0 = sortIx[sortIx >= numItems]
        i, j = df0.index, df0.values - numItems
        sortIx[i] = link[j, 0]
        df0 = pd.Series(link[j, 1], index=i + 1)
        sortIx = pd.concat([sortIx, df0]).sort_index()
        sortIx.index = range(sortIx.shape[0])
    return sortIx.tolist()

def getRecBipart(cov, sortIx):
    w = pd.Series(1.0, index=sortIx)
    cItems = [sortIx]
    # Split the ordered asset list into halves, not the tree's child branches.
    while cItems:
        cItems = [i[j:k] for i in cItems for j, k in
                  ((0, len(i) // 2), (len(i) // 2, len(i))) if len(i) > 1]
        for i in range(0, len(cItems), 2):
            cVar0 = getClusterVar(cov, cItems[i])
            cVar1 = getClusterVar(cov, cItems[i + 1])
            alpha = 1 - cVar0 / (cVar0 + cVar1)
            w[cItems[i]] *= alpha
            w[cItems[i + 1]] *= 1 - alpha
    return w

### 2.2 Distance measures

Each function returns a symmetric asset-by-asset matrix. Let **T** denote the number of return observations and **N** the number of assets.

| Key | Implemented measure |
|---|---|
| d1 | Square root of (1 − correlation) / 2 |
| d2 | (1 − correlation) / 2 |
| d3 | Histogram variation of information: 2H(X,Y) − H(X) − H(Y) |
| d4 | Mean absolute difference: sum(abs(x − y)) / T |
| d5 | Root mean squared difference: sqrt(sum((x − y)²) / T) |
| d6 | Maximum absolute difference divided by T |

The T scaling is part of this implementation. VI uses raw entropy in natural-log units with `max(10, int(sqrt(T)))` bins. Binning and normalization should be documented when comparing with the source paper.


In [ ]:
# Build one asset-distance matrix per measure from the supplied return sample.
# T is the number of observations; N is the number of assets.

def d1_matrix(returns):
    """d1 = sqrt(0.5*(1-rho))  — standard HRP (eq. 7)"""
    rho = returns.corr().values
    return np.sqrt(np.clip(0.5 * (1 - rho), 0, None))

def d2_matrix(returns):
    """d2 = (1-rho)/2  — linear correlation (eq. 8)"""
    rho = returns.corr().values
    return np.clip((1 - rho) / 2, 0, None)

def _vi(x, y, bins):
    """Variation of information VI[X,Y] = H[X,Y] - I[X,Y]."""
    cXY, _, _ = np.histogram2d(x, y, bins=bins)
    pXY = cXY / cXY.sum()
    pX  = pXY.sum(axis=1, keepdims=True)
    pY  = pXY.sum(axis=0, keepdims=True)
    hX  = -np.sum(pX  * np.log(pX  + 1e-10))
    hY  = -np.sum(pY  * np.log(pY  + 1e-10))
    hXY = -np.sum(pXY * np.log(pXY + 1e-10))
    return max(hXY - (hX + hY - hXY), 0.0)  # VI = H[X,Y] - I[X,Y]

def d3_matrix(returns):
    """d3 = VI[X,Y] — variation of information (eq. 9).

    Implementation notes (paper does not specify either):
    - Raw (unnormalised) VI is used. De Prado (2020) sometimes uses
      normalised VI = VI[X,Y]/H[X,Y] in [0,1]; if the paper uses that
      form, d3 results will differ systematically.
    - Bin count: max(10, sqrt(T)) by the square-root rule of thumb.
      Entropy estimates are sensitive to bin count; paper does not state
      a value, so this is an unverifiable source of divergence from Table 2.
    """
    arr  = returns.values
    n    = arr.shape[1]
    bins = max(10, int(np.sqrt(len(returns))))  # rule of thumb; paper unspecified
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            v = _vi(arr[:, i], arr[:, j], bins)
            D[i, j] = D[j, i] = v
    return D

def d4_matrix(returns):
    """d4 = (1/T)*sum|xi-yi|  — Manhattan (eq. 11)"""
    arr = returns.values
    n, T = arr.shape[1], arr.shape[0]
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            v = np.sum(np.abs(arr[:, i] - arr[:, j])) / T
            D[i, j] = D[j, i] = v
    return D

def d5_matrix(returns):
    """d5 = sqrt((1/T)*sum(xi-yi)^2)  — Euclidean (eq. 12)"""
    arr = returns.values
    n, T = arr.shape[1], arr.shape[0]
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            v = np.sqrt(np.sum((arr[:, i] - arr[:, j])**2) / T)
            D[i, j] = D[j, i] = v
    return D

def d6_matrix(returns):
    """d6 = (1/T)*max|xi-yi|  — Chebyshev (eq. 13)"""
    arr = returns.values
    n, T = arr.shape[1], arr.shape[0]
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            v = np.max(np.abs(arr[:, i] - arr[:, j])) / T
            D[i, j] = D[j, i] = v
    return D

DIST_FUNCS = {'d1': d1_matrix, 'd2': d2_matrix, 'd3': d3_matrix,
              'd4': d4_matrix, 'd5': d5_matrix, 'd6': d6_matrix}


### 2.3 HRP wrapper and benchmark methods

`getHRP` builds the selected distance, clusters assets and returns HRP weights. The other methods return weights in the same asset order:

- **IVP:** inverse individual variance.
- **Quadratic / MVO:** long-only minimum variance fitted with SLSQP.
- **ESR:** approximate in-sample Sharpe maximization using differential evolution, a sum-to-one penalty and final normalization.
- **Equal weight:** 1/N for every asset.

The optimizer result flags are not checked by the current implementation. ESR is an approximation of the method cited in the original replication notes.


In [ ]:
# Fit each portfolio method and return weights in the input asset order.
# The optimizer success/status fields are not checked here.

def getHRP(returns, dist_key):
    D = DIST_FUNCS[dist_key](returns)
    np.fill_diagonal(D, 0)
    D = (D + D.T) / 2
    link   = sch.linkage(squareform(D), 'single')
    sortIx = getQuasiDiag(link)
    cols   = returns.columns
    sortIx = [cols[i] for i in sortIx]
    cov    = pd.DataFrame(returns.cov().values, index=cols, columns=cols)
    return getRecBipart(cov, sortIx).reindex(returns.columns).values

def getMVO(returns):
    """Minimum variance portfolio (scipy SLSQP; paper uses quadratic optimizer)."""
    n   = returns.shape[1]
    cov = returns.cov().values
    res = minimize(
        fun=lambda w: float(np.dot(w, np.dot(cov, w))),
        x0=np.ones(n) / n,
        method='SLSQP',
        bounds=[(0, 1)] * n,
        constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1},
        options={'ftol': 1e-9, 'maxiter': 500}
    )
    # Return the candidate solution; res.success is not inspected.
    return res.x

def getIVP_w(returns):
    cov = returns.cov().values
    ivp = 1.0 / np.diag(cov)
    return ivp / ivp.sum()

def getESR(returns):
    """Evolutionary Sharpe Ratio — approximated with differential evolution."""
    n   = returns.shape[1]
    cov = returns.cov().values
    mu  = returns.mean().values
    def neg_sharpe(w):
        p_vol = np.sqrt(max(np.dot(w, np.dot(cov, w)), 1e-12))
        return -np.dot(w, mu) / p_vol + 1e4 * (w.sum() - 1)**2
    # Fixed seed for the optimizer; input data and column order must also be fixed.
    res = differential_evolution(neg_sharpe, [(0, 1)] * n,
                                 seed=42, maxiter=100, popsize=10, tol=1e-5)
    w = np.maximum(res.x, 0)
    return w / w.sum()

def getEW(returns):
    n = returns.shape[1]
    return np.ones(n) / n

## 3. Contiguous-fold evaluation

* Divide each scenario into five contiguous held-out blocks.
* For each block, fit the portfolio using **all other blocks**, then compare its training and held-out mean and variance.
* Training can therefore include dates later than the held-out block; this is not walk-forward validation.

* For each method, average the squared train–test discrepancies.
* Divide return MSE by index return MSE, and variance MSE by index variance MSE.
* The combined score is the Euclidean norm of those two NMSE values.

* A value below 1 means less estimation discrepancy than the index on that dimension.
* It does not by itself establish a higher return, lower realized risk or a higher Sharpe ratio.


In [ ]:
# Hold out one contiguous block and train on every other block.
# Measure train–test moment discrepancies relative to the index.

def port_stats(ret_arr, w):
    port = ret_arr @ w
    # np.var uses ddof=0; the index calculations below use pandas ddof=1.
    return np.mean(port), np.var(port)

def kfold_eval(returns, index_returns, k=K_FOLDS):
    """
    For each fold i:
      fit portfolio on k-1 training folds
      ŷᵢ = in-sample mean/var (training data)
      yᵢ = OOS mean/var (test fold)
    MSE(Aj) = (1/k)Σ(ŷᵢ - yᵢ)²
    NMSE(Aj) = MSE(Aj) / MSE(A₀=IBEX)
    """
    # Align index dates and fill gaps; bfill may use a later observation.
    index_returns = index_returns.reindex(returns.index).ffill().bfill()
    T = len(returns)
    # The last block receives any remainder after integer division.
    fb = [i * (T // k) for i in range(k)] + [T]

    methods = list(DIST_FUNCS.keys()) + ['IVP', 'MVO', 'ESR', 'EW']
    rec = {m: dict(pr=[], pv=[], ar=[], av=[]) for m in methods + ['IDX']}

    for fi in range(k):
        mask = np.zeros(T, dtype=bool)
        mask[fb[fi]:fb[fi+1]] = True

        # Complement training includes earlier and later dates around the test block.
        rtr, rte = returns.iloc[~mask], returns.iloc[mask]
        itr, ite = index_returns.iloc[~mask], index_returns.iloc[mask]

        rec['IDX']['pr'].append(itr.mean());  rec['IDX']['pv'].append(itr.var())
        rec['IDX']['ar'].append(ite.mean());  rec['IDX']['av'].append(ite.var())

        for dk in DIST_FUNCS:
            w = getHRP(rtr, dk)
            pr, pv = port_stats(rtr.values, w)
            ar, av = port_stats(rte.values, w)
            rec[dk]['pr'].append(pr); rec[dk]['pv'].append(pv)
            rec[dk]['ar'].append(ar); rec[dk]['av'].append(av)

        for key, fn in [('IVP', getIVP_w), ('MVO', getMVO), ('ESR', getESR), ('EW', getEW)]:
            w = fn(rtr)
            pr, pv = port_stats(rtr.values, w)
            ar, av = port_stats(rte.values, w)
            rec[key]['pr'].append(pr); rec[key]['pv'].append(pv)
            rec[key]['ar'].append(ar); rec[key]['av'].append(av)

    def mse(p, a): return np.mean([(x-y)**2 for x, y in zip(p, a)])

    # These normalization denominators are not guarded against zero.
    m0r = mse(rec['IDX']['pr'], rec['IDX']['ar'])
    m0v = mse(rec['IDX']['pv'], rec['IDX']['av'])

    labels = {dk: f'HRP-{dk}' for dk in DIST_FUNCS}
    labels.update({'IVP': 'IVP', 'MVO': 'Quadratic', 'ESR': 'ESR', 'EW': 'Equal-weight'})

    rows = {}
    for key, label in labels.items():
        nr = mse(rec[key]['pr'], rec[key]['ar']) / m0r
        nv = mse(rec[key]['pv'], rec[key]['av']) / m0v
        rows[label] = {'Return': round(nr, 2), 'Risk': round(nv, 2),
                       'Euclidean': round(np.sqrt(nr**2 + nv**2), 2)}

    # Average the already rounded HRP scores.
    hrp_rows = [rows[f'HRP-d{i}'] for i in range(1, 7)]
    rows['HRP average'] = {
        k: round(np.mean([r[k] for r in hrp_rows]), 2) for k in ['Return', 'Risk', 'Euclidean']}

    return pd.DataFrame(rows).T[['Return', 'Risk', 'Euclidean']]

## 4. Run the scenarios

* Evaluate every HRP distance and benchmark in each available scenario.
* Store one NMSE table per scenario in `all_results` and print the elapsed time. Empty datasets are skipped.


In [ ]:
# Run all portfolio methods for each available scenario and time the evaluation.

import time
all_results = {}
for name in SCENARIOS:
    ret = scenario_returns[name]
    if ret.empty or ret.shape[1] == 0:
        print(f'{name}: no data — skipping')
        continue
    print(f'Running {name} ({ret.shape[1]} assets, k={K_FOLDS})...', flush=True)
    t0 = time.time()
    all_results[name] = kfold_eval(ret, scenario_index[name])
    print(f'  Done in {time.time()-t0:.1f}s')

## 5. Compare with the reference table

* The `paper` dictionary contains manually transcribed reference values.
* Print them beside the calculated results, matching by scenario and method.
* The transcription is preserved; it is not automatically checked against the publication.

* The evaluation function rounds results before returning them, including components used by the HRP average.
* Keep that convention in mind when comparing rounded values.


In [ ]:
# Compare calculated tables with the manually transcribed reference values.

paper = {
    'Bull': {
        'HRP-d1':(1.01,0.58,1.16),'HRP-d2':(1.00,0.56,1.15),'HRP-d3':(1.04,0.58,1.19),
        'HRP-d4':(1.09,0.77,1.33),'HRP-d5':(1.07,0.84,1.36),'HRP-d6':(1.06,0.72,1.28),
        'HRP average':(1.05,0.68,1.25),'Quadratic':(1.03,0.60,1.32),
        'IVP':(1.09,0.84,1.38),'ESR':(1.12,0.85,1.41),'Equal-weight':(1.36,1.27,1.86),
    },
    'Bear': {
        'HRP-d1':(0.64,0.34,0.72),'HRP-d2':(0.62,0.36,0.72),'HRP-d3':(0.65,0.35,0.74),
        'HRP-d4':(0.74,0.43,0.86),'HRP-d5':(0.73,0.44,0.85),'HRP-d6':(0.65,0.41,0.77),
        'HRP average':(0.67,0.39,0.78),'Quadratic':(0.52,0.30,0.60),
        'IVP':(0.75,0.49,0.90),'ESR':(0.66,0.25,0.71),'Equal-weight':(1.16,0.64,1.32),
    },
    'Sideways': {
        'HRP-d1':(0.74,0.67,1.00),'HRP-d2':(0.83,0.67,1.07),'HRP-d3':(0.76,0.77,1.08),
        'HRP-d4':(0.83,0.79,1.15),'HRP-d5':(0.80,0.77,1.11),'HRP-d6':(0.79,0.76,1.10),
        'HRP average':(0.79,0.74,1.08),'Quadratic':(0.76,0.66,1.01),
        'IVP':(0.77,0.78,1.10),'ESR':(1.46,0.72,1.63),'Equal-weight':(1.31,1.20,1.78),
    },
}

for name, df in all_results.items():
    paper_df = pd.DataFrame(paper[name], index=['Return','Risk','Euclidean']).T
    print(f'\n══ {name} ══')
    print(f'{"Method":<15} {"Ret(ours)":>10} {"Ret(paper)":>11} {"Risk(ours)":>11} {"Risk(paper)":>12} {"Euc(ours)":>10} {"Euc(paper)":>11}')
    print('-' * 85)
    for method in df.index:
        r  = df.loc[method]
        p  = paper[name].get(method, (None, None, None))
        pr = f'{p[0]:.2f}' if p[0] else '—'
        pv = f'{p[1]:.2f}' if p[1] else '—'
        pe = f'{p[2]:.2f}' if p[2] else '—'
        print(f'{method:<15} {r["Return"]:>10.2f} {pr:>11} {r["Risk"]:>11.2f} {pv:>12} {r["Euclidean"]:>10.2f} {pe:>11}')

### 5.1 Return–risk NMSE plot

* Plot calculated and reference NMSE values in separate panels.
* Colour identifies the scenario and marker shape identifies the method.
* Include the HRP average in both panels.
* Dashed lines at 1 mark the index-relative threshold. Each panel has its own axis scale so outliers remain visible without compressing the reference panel.


In [ ]:
# Compare calculated and reference NMSE with consistent methods and visual labels.
# Colour identifies the scenario; marker shape identifies the portfolio method.
from matplotlib.lines import Line2D

method_markers = {
    'HRP-d1': 'o', 'HRP-d2': 's', 'HRP-d3': '^',
    'HRP-d4': 'v', 'HRP-d5': '<', 'HRP-d6': '>',
    'HRP average': 'h', 'Quadratic': 'D', 'IVP': 'P',
    'ESR': 'X', 'Equal-weight': '*',
}
scenario_colors = {'Bull': '#1f77b4', 'Bear': '#ff7f0e', 'Sideways': '#2ca02c'}
reference_results = {
    name: pd.DataFrame(values, index=['Return', 'Risk', 'Euclidean']).T
    for name, values in paper.items()
}

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, title, tables in zip(
    axes,
    ['Calculated results', 'Reference: Table 2'],
    [all_results, reference_results],
):
    for scenario, color in scenario_colors.items():
        table = tables.get(scenario)
        if table is None:
            continue
        for method, marker in method_markers.items():
            if method not in table.index:
                continue
            ax.scatter(
                table.loc[method, 'Return'], table.loc[method, 'Risk'],
                marker=marker, color=color, s=75, alpha=0.8,
            )
    ax.set(title=title, xlabel='Return NMSE', ylabel='Variance NMSE')
    ax.axhline(1, color='grey', linewidth=0.8, linestyle='--')
    ax.axvline(1, color='grey', linewidth=0.8, linestyle='--')
    ax.grid(alpha=0.15)

# Separate legends make both visual encodings explicit.
scenario_handles = [
    Line2D([0], [0], marker='o', linestyle='none', color=color, label=name)
    for name, color in scenario_colors.items()
]
method_handles = [
    Line2D([0], [0], marker=marker, linestyle='none', color='#444444', label=name)
    for name, marker in method_markers.items()
]
axes[0].legend(handles=scenario_handles, title='Scenario', fontsize=9)
fig.legend(
    handles=method_handles, title='Portfolio method', loc='lower center',
    ncol=6, fontsize=9, bbox_to_anchor=(0.5, 0.01),
)
fig.suptitle('Salas-Molina replication: normalized estimation errors', fontsize=13)
fig.tight_layout(rect=(0, 0.17, 1, 0.95))
fig.savefig('salas_molina_nmse.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: salas_molina_nmse.png')
